# Figure 1 and 2
This notebook produces the plots in Figures 1 and 2 in [Ronchi et al. 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...916..100R/abstract).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import typing
import scipy as sc
from scipy import stats

import utilities.plot_settings
import mlpoppyns.generator.maps.axes_scaling as axs

In [ ]:
def generate_density_map(
    x: np.ndarray,
    x_range: typing.Tuple[float, float],
    y: np.ndarray,
    y_range: typing.Tuple[float, float],
    filename: str,
    figsize: typing.Tuple[float, float] = (8., 6.),
    x_log_scale: bool = False,
    y_log_scale: bool = False,
    n_x_bins: int = 128,
    n_y_bins: int = 128,
    colormap: str = "jet",
) -> None:
    """
    Density map generator.

    Creates a density or heat map of a distribution of points given their
    X/Y coordinates in a 2D space. The resulting image is written to disk.

    Args:
        x (np.ndarray): horizontal coordinate values for the points.
        x_range (float, float): horizontal range of values for the points.
        y (np.ndarray): vertical coordinate values for the points.
        y_range (float, float): vertical range of values for the points.
        filename (str): file path to generate the density map image.
        x_log_scale (bool): if True set the x axis scale to log scale.
        y_log_scale (bool): if True set the y axis scale to log scale.
        n_x_bins (int): number of horizontal bins for the density map.
        n_y_bins (int): number of vertical bins for the density map.
        colormap (str): colormap to use for the image.

    Returns:
        Nothing. An image is generated in the specified file path.

    """

    x_edges, y_edges = axs.log_scale_vs_linear_scale(
        x_range, y_range, x_log_scale, y_log_scale, n_x_bins, n_y_bins,
    )

    # Generating a 2D histogram that counts the number of objects contained
    # in each respective pixel; following the discrete count, we apply a
    # Gaussian filter to smear out the hard edges of the distribution to
    # improve the stability of the machine learning framework;
    # x (y) values are histogrammed along first (second) dimension.
    density, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    density = sc.ndimage.gaussian_filter(density, sigma=1)
    print(np.max(density))

    fig, ax = plt.subplots(figsize=figsize)

    # Generating a pseudocolor plot of the smeared out density distribution;
    # we transpose the array as pcolormesh is indexed starting from the lower
    # left , i.e., the column (row) index corresponds to the x (y) coordinate.
    pcm=ax.pcolormesh(x_edges, y_edges, density.T, cmap=colormap, rasterized=True)
    ax.set_xlim(x_range[0], x_range[1])
    ax.set_ylim(y_range[0], y_range[1])

    if x_log_scale:
        ax.set_xscale("log")

    if y_log_scale:
        ax.set_yscale("log")

    ax.axes.xaxis.set_visible(False)
    ax.axes.yaxis.set_visible(False)
    # ax.set_xlabel(r"RA [deg]")
    # ax.set_ylabel(r"DEC [deg]")
    cbar=fig.colorbar(pcm)
    cbar.set_label(r'Number of NS')
    #fig.savefig(filename, dpi=DPI)
    #plt.close(fig)


def generate_avg_weight_map(
    x: np.ndarray,
    x_range: typing.Tuple[float, float],
    y: np.ndarray,
    y_range: typing.Tuple[float, float],
    w: np.ndarray,
    filename: str,
    vrange: typing.Tuple[float, float],
    bar_label: str,
    figsize: typing.Tuple[float, float] = (8., 6.),
    x_log_scale: bool = False,
    y_log_scale: bool = False,
    n_x_bins: int = 128,
    n_y_bins: int = 128,
    colormap: str = "jet",
    
) -> None:
    """
    Average weighted map generator.

    Creates a map of the average weight w of a distribution of points given their
    X/Y coordinates in a 2D space.
    The resulting map shows the average value of the weights in each bin.

    Args:
        x (np.ndarray): horizontal coordinate values for the points.
        x_range (float, float): horizontal range of values for the points.
        y (np.ndarray): vertical coordinate values for the points.
        y_range (float, float): vertical range of values for the points.
        w (np.ndarray): weight values for the points.
        filename (str): file path to generate the heat map image.
        vrange (float, float): range of values for the colormap.
        bar_label (str): label of the colorbar.
        figsize (float, float): size of the figure.
        x_log_scale (bool): if True set the x axis scale to log scale.
        y_log_scale (bool): if True set the y axis scale to log scale.
        n_x_bins (int): number of horizontal bins for the weight map.
        n_y_bins (int): number of vertical bins for the weight map.
        colormap (str): colormap to use for the image.
        
    Returns:
        Nothing. An image is generated in the specified file path.

    """

    x_edges, y_edges = axs.log_scale_vs_linear_scale(
        x_range, y_range, x_log_scale, y_log_scale, n_x_bins, n_y_bins,
    )

    # If the quantity desired as the weight can become negative, e.g.,
    # one of the velocity components, take the absolute value and use that
    # as the weight to avoid the possibility of summing to zero.
    total_per_bin, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    total_weight, x_edges, y_edges = np.histogram2d(
        x, y, bins=[x_edges, y_edges], weights=w
    )

    # Dividing the total summed weight per bin by the number of objects to
    # obtain the average value per pixel; to avoid dividing by 0, we change
    # the values in total_weight from 0 to 0.0001; doing so does not affect
    # the final result as the original array elements are zero anyway;
    # to avoid potential sharp edges, we apply a Gaussian filter.
    total_per_bin[total_per_bin == 0] = 0.0001
    avg_weight = total_weight / total_per_bin
    avg_weight = sc.ndimage.gaussian_filter(avg_weight, sigma=1)
    print(np.max(avg_weight))

    fig, ax = plt.subplots(figsize=figsize)

    # Generating a pseudocolor plot of the smeared out average distribution;
    # we transpose the array as pcolormesh is indexed starting from the lower
    # left , i.e., the column (row) index corresponds to the x (y) coordinate.
    pcm=ax.pcolormesh(x_edges, y_edges, avg_weight.T, cmap=colormap, vmin=vrange[0], vmax=vrange[1], rasterized=True)
    ax.set_xlim(x_range[0], x_range[1])
    ax.set_ylim(y_range[0], y_range[1])

    if x_log_scale:
        ax.set_xscale("log")

    if y_log_scale:
        ax.set_yscale("log")

    ax.axes.xaxis.set_visible(False)
    ax.axes.yaxis.set_visible(False)
    #ax.set_xlabel(r"RA [deg]")
    #ax.set_ylabel(r"DEC [deg]")
    cbar=fig.colorbar(pcm)
    cbar.set_label(bar_label)
    #fig.savefig(filename, dpi=DPI)
    #plt.close(fig)

In [ ]:
data = pd.read_pickle("../../data/paper_results/ronchi_etal_2021/simulation_maxwell_265/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
v_r = data["v_r"]["[km/s]"].to_numpy()
v_phi = data["v_phi"]["[km/s]"].to_numpy()
v_z = data["v_z"]["[km/s]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
v_RA = data["v_RA"]["[mas/yr]"].to_numpy()
v_DEC = data["v_DEC"]["[mas/yr]"].to_numpy()

In [ ]:
generate_density_map(
    x=x, 
    x_range=(-20., 20.), 
    y=y, 
    y_range=(-20., 20.),
    filename='xy_position_map',
    n_x_bins=128,
    n_y_bins= 128
)

plt.savefig(
    f"plots/Figure1_a.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_density_map(
    x=x, 
    x_range=(-20., 20.), 
    y=z, 
    y_range=(-20., 20.),
    filename='xz_position_map',
    n_x_bins=128,
    n_y_bins= 128
)

In [ ]:
generate_avg_weight_map(
    x=x, 
    x_range=(-20., 20.), 
    y=y, 
    y_range=(-20., 20.),
    w=abs(v_r),
    filename='vr_velocity_map',
    n_x_bins=128,
    n_y_bins= 128,
    vrange=(0, 425.),
    bar_label = r'$\overline{\left| v_{r} \right|}$ [km s$^{-1}$]'
)

plt.savefig(
    f"plots/Figure1_b.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_avg_weight_map(
    x=x, 
    x_range=(-20., 20.), 
    y=y, 
    y_range=(-20., 20.),
    w=abs(v_phi),
    filename='vphi_velocity_map',
    n_x_bins=128,
    n_y_bins= 128,
    vrange=(0, 425.),
    bar_label = r'$\overline{\left| v_{\phi} \right|}$ [km s$^{-1}$]'
)

plt.savefig(
    f"plots/Figure1_c.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_avg_weight_map(
    x=x, 
    x_range=(-20., 20.), 
    y=y, 
    y_range=(-20., 20.),
    w=abs(v_z),
    filename='vz_velocity_map',
    n_x_bins=128,
    n_y_bins= 128,
    vrange=(0, 425.),
    bar_label = r'$\overline{\left| v_{z} \right|}$ [km s$^{-1}$]'
)

plt.savefig(
    f"plots/Figure1_d.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_density_map(
    x=RA, 
    x_range=(0., 360.), 
    y=DEC, 
    y_range=(-90., 90.),
    filename='radec_position_map',
    figsize=(12, 6),
    n_x_bins=128,
    n_y_bins= 64
)

plt.savefig(
    f"plots/Figure2_a.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_avg_weight_map(
    x=RA, 
    x_range=(0., 360.), 
    y=DEC, 
    y_range=(-90., 90.),
    w=abs(v_RA),
    filename='vra_velocity_map',
    figsize=(12, 6),
    n_x_bins=128,
    n_y_bins= 64,
    vrange=(0., 75.),
    bar_label = r'$\overline{\left| \mu_{\rm RA} \right|}$ [mas yr$^{-1}$]'
)

plt.savefig(
    f"plots/Figure2_b.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
generate_avg_weight_map(
    x=RA, 
    x_range=(0., 360.), 
    y=DEC, 
    y_range=(-90., 90.),
    w=abs(v_DEC),
    filename='vdec_velocity_map',
    figsize=(12, 6),
    n_x_bins=128,
    n_y_bins= 64,
    vrange=(0., 75.),
    bar_label = r'$\overline{\left| \mu_{\rm DEC} \right|}$ [mas yr$^{-1}$]'
)

plt.savefig(
    f"plots/Figure2_c.pdf",
    dpi=300,
    bbox_inches="tight",
)